# Monte Carlo Expected Wins - Uncertainty Visualization

## Demonstration of Confidence Interval Display

This notebook shows how to visualize expected wins with uncertainty bands.

**Example Output**:
```
Team: bplenzen
Actual wins: 2
Expected wins: 3.5 ± 0.7 (2.8 - 4.2)
Luck: -1.5 wins (likely unlucky, but within expected variance)
```

**Goal**: Transform "Expected Wins: 5.2" → "Expected Wins: 5.2 ± 0.7 (4.5 - 5.9)"

---

In [ ]:
import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Connect to warehouse
conn = duckdb.connect('../data/warehouse.duckdb', read_only=True)

# Load luck data with uncertainty
luck_df = conn.execute("""
    SELECT 
        manager_name,
        actual_wins,
        expected_wins_p50 as expected_wins,
        expected_wins_p05 as ci_low,
        expected_wins_p95 as ci_high,
        expected_wins_ci_width as ci_width,
        wins_over_expected_p50 as woe,
        composite_luck_score
    FROM main_analytics.fct_advanced_luck
    ORDER BY expected_wins DESC
""").df()

print(f"Loaded {len(luck_df)} teams")
luck_df.head()

## Fan Chart: Actual vs Expected Wins with Uncertainty Bands

Shows where each team falls relative to their expected win range.

In [ ]:
# Sort by expected wins for better visualization
luck_df_sorted = luck_df.sort_values('expected_wins')

fig = go.Figure()

# Add CI bands (shaded area)
fig.add_trace(go.Scatter(
    x=luck_df_sorted['manager_name'],
    y=luck_df_sorted['ci_high'],
    mode='lines',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=luck_df_sorted['manager_name'],
    y=luck_df_sorted['ci_low'],
    mode='lines',
    line=dict(width=0),
    fillcolor='rgba(68, 138, 255, 0.2)',
    fill='tonexty',
    name='95% Confidence Interval',
    hovertemplate='%{y:.1f} wins<extra></extra>'
))

# Add expected wins line
fig.add_trace(go.Scatter(
    x=luck_df_sorted['manager_name'],
    y=luck_df_sorted['expected_wins'],
    mode='lines+markers',
    name='Expected Wins (p50)',
    line=dict(color='blue', width=2),
    marker=dict(size=8),
    hovertemplate='Expected: %{y:.1f} wins<extra></extra>'
))

# Add actual wins
fig.add_trace(go.Scatter(
    x=luck_df_sorted['manager_name'],
    y=luck_df_sorted['actual_wins'],
    mode='markers',
    name='Actual Wins',
    marker=dict(size=12, color='red', symbol='diamond'),
    hovertemplate='Actual: %{y} wins<extra></extra>'
))

fig.update_layout(
    title='Actual vs Expected Wins with 95% Confidence Intervals',
    xaxis_title='Manager',
    yaxis_title='Wins',
    hovermode='x unified',
    template='plotly_white',
    height=500,
    legend=dict(x=0.01, y=0.99)
)

fig.show()

print("\n**Interpretation**:")
print("- Blue line: Expected wins based on all-play performance")
print("- Shaded area: 95% confidence interval (expected variance)")
print("- Red diamonds: Actual wins")
print("- Inside shaded area: Within expected luck variance")
print("- Outside shaded area: Statistically significant luck (good or bad)")

## Formatted Luck Report with Uncertainty

Shows how to display the uncertainty in text format.

In [ ]:
print("=" * 80)
print("LUCK ANALYSIS WITH UNCERTAINTY QUANTIFICATION")
print("=" * 80)

for _, row in luck_df.iterrows():
    manager = row['manager_name']
    actual = int(row['actual_wins'])
    expected = row['expected_wins']
    ci_low = row['ci_low']
    ci_high = row['ci_high']
    ci_width = row['ci_width']
    woe = row['woe']
    
    # Determine luck interpretation
    if actual < ci_low:
        luck_status = "❌ UNLUCKY (below 95% CI)"
    elif actual > ci_high:
        luck_status = "🍀 LUCKY (above 95% CI)"
    else:
        luck_status = "✅ Within expected variance"
    
    print(f"\n{manager:15s}")
    print(f"  Actual wins:    {actual}")
    print(f"  Expected wins:  {expected:.1f} ± {ci_width/2:.1f}  ({ci_low:.1f} - {ci_high:.1f})")
    print(f"  Wins over exp:  {woe:+.1f}")
    print(f"  {luck_status}")

print("\n" + "=" * 80)
print("KEY:")
print("  ± value = half of 95% confidence interval width")
print("  (range) = 95% confidence interval for expected wins")
print("  ✅ = Actual wins within statistical expected range")
print("  🍀 = Significantly luckier than expected (p < 0.05)")
print("  ❌ = Significantly unluckier than expected (p < 0.05)")
print("=" * 80)

In [ ]:
# Monte Carlo Expected Wins - Uncertainty Visualization

## Demonstration of Confidence Interval Display

This notebook shows how to visualize expected wins with uncertainty bands.

**Example Output**:
```
Team: bplenzen
Actual wins: 2
Expected wins: 3.5 ± 0.7 (2.8 - 4.2)
Luck: -1.5 wins (within expected range)
```

---